# **LTX-VIDEO Text to Video**

- You can use the free T4 GPU to run this depending on the output video resolution and number of frames. The default setting runs without issues, but at 768 by 512 output resolution with 121 frames, the decoding process crashes the 12.7GB RAM.  For faster video generation with higher resolutions and frames, use higher GPUs.
- If you want to generate a video with n frames, then set frames to n+1. e.g. To generate a video with 72 frames, set frames to 73.
- You need to use detailed prompts to get decent results.
- Videos are generated at 24fps.

In [ ]:
# @title Prepare Environment
!pip install torch==2.6.0 torchvision==0.21.0
%cd /content
Always_Load_Models_for_Inference = False
Use_t5xxl_fp16 = False
# Install dependencies
!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2
!pip install av
!git clone https://github.com/Isi-dev/ComfyUI
%cd /content/ComfyUI
!apt -y install -qq aria2 ffmpeg

# Download required models
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Isi99999/LTX-Video/resolve/main/ltx-video-2b-v0.9.5.safetensors -d /content/ComfyUI/models/checkpoints -o ltx-video-2b-v0.9.5.safetensors
if Use_t5xxl_fp16:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Isi99999/LTX-Video/resolve/main/t5xxl_fp16.safetensors -d /content/ComfyUI/models/text_encoders -o t5xxl_fp16.safetensors
else:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Isi99999/LTX-Video/resolve/main/t5xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o t5xxl_fp8_e4m3fn_scaled.safetensors

# Initial setup
import torch
import numpy as np
from PIL import Image
import gc
import sys
import random
import os
import imageio
from google.colab import files
from IPython.display import display, HTML
sys.path.insert(0, '/content/ComfyUI')

from comfy import model_management

from nodes import (
    CheckpointLoaderSimple,
    CLIPLoader,
    CLIPTextEncode,
    VAEDecode
)

from comfy_extras.nodes_custom_sampler import (
    KSamplerSelect,
    SamplerCustom
)

from comfy_extras.nodes_lt import (
    LTXVConditioning,
    LTXVScheduler,
    EmptyLTXVLatentVideo
)

checkpoint_loader = CheckpointLoaderSimple()
clip_loader = CLIPLoader()
clip_encode_positive = CLIPTextEncode()
clip_encode_negative = CLIPTextEncode()
scheduler = LTXVScheduler()
sampler_select = KSamplerSelect()
conditioning = LTXVConditioning()
empty_latent_video = EmptyLTXVLatentVideo()
sampler = SamplerCustom()
vae_decode = VAEDecode()

# if not Always_Load_Models_for_Inference:
# with torch.inference_mode():
#     # Load models
#     print("Loading Model...")
#     model, _, vae = checkpoint_loader.load_checkpoint("ltx-video-2b-v0.9.5.safetensors")
#     print("Loaded model!")
#     # print("Loading Text_Encoder...")
#     # clip = clip_loader.load_clip("t5xxl_fp8_e4m3fn_scaled.safetensors", "ltxv", "default")[0]
#     # print("Loaded Text_Encoder!")


def clear_memory():
    """Frees GPU (VRAM) and CPU RAM memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    for obj in list(globals().values()):
        if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
            del obj

    gc.collect()

def generate_video(
    positive_prompt: str = "A drone quickly rises through a bank of morning fog...",
    negative_prompt: str = "low quality, worst quality...",
    width: int = 768,
    height: int = 512,
    seed: int = 0,
    steps: int = 30,
    cfg_scale: float = 2.05,
    sampler_name: str = "res_multistep",
    length: int = 49,
    fps: int = 24
):

    with torch.inference_mode():
        print("Loading Text_Encoder...")
        clip = clip_loader.load_clip("t5xxl_fp8_e4m3fn_scaled.safetensors", "ltxv", "default")[0]
        print("Loaded Text_Encoder!")

    try:
        assert width % 32 == 0, "Width must be divisible by 32"
        assert height % 32 == 0, "Height must be divisible by 32"

        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]

        del clip
        torch.cuda.empty_cache()
        gc.collect()
        print("Text_Encoder removed from memory")

        empty_latent = empty_latent_video.generate(width, height, length)[0]

        sigmas = scheduler.get_sigmas(steps, cfg_scale, 0.95, True, 0.1)[0]
        selected_sampler = sampler_select.get_sampler(sampler_name)[0]
        conditioned = conditioning.append(positive, negative, 25.0)

        print("Loading model & VAE...")
        model, _, vae = checkpoint_loader.load_checkpoint("ltx-video-2b-v0.9.5.safetensors")
        print("Loaded model & VAE!")

        print("Generating video...")
        sampled = sampler.sample(
            model=model,
            add_noise=True,
            noise_seed=seed if seed != 0 else random.randint(0, 2**32),
            cfg=cfg_scale,
            positive=conditioned[0],
            negative=conditioned[1],
            sampler=selected_sampler,
            sigmas=sigmas,
            latent_image=empty_latent
        )[0]

        del model
        torch.cuda.empty_cache()
        gc.collect()
        print("Model removed from memory")

        with torch.no_grad():
          try:
              print("Decodimg Latents...")
              decoded = vae_decode.decode(vae, sampled)[0].detach()
              print("Latents Decoded!")
              del vae
              torch.cuda.empty_cache()
              gc.collect()
              print("VAE removed from memory")

              output_path = "/content/output.mp4"
              with imageio.get_writer(output_path, fps=fps) as writer:
                  for i, frame in enumerate(decoded):
                      frame_np = (frame.cpu().numpy() * 255).astype(np.uint8)
                      writer.append_data(frame_np)
                      if i % 10 == 0:  # Periodic cleanup
                          torch.cuda.empty_cache()

              print(f"Successfully processed {len(decoded)} frames")


          except Exception as e:
              print(f"Decoding error: {str(e)}")
              raise

        print("Displaying Video...")
        display_video(output_path)

    except Exception as e:
        print(f"Video generation failed: {str(e)}")
        raise
    finally:
        clear_memory()

def display_video(video_path):
    from IPython.display import HTML
    from base64 import b64encode

    mp4 = open(video_path,'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    display(HTML(f"""
    <video width=512 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    """))

print("✅ Environment Setup Complete!")

In [ ]:
# @title Video Generation Parameters
# example_prompt = "A cinematic aerial view from a slowly moving drone, capturing breathtaking landscapes. The camera smoothly glides over rolling green hills, vast forests, and shimmering lakes, bathed in the golden light of sunrise. Mist gently rises from the valleys, creating a dreamy atmosphere. The drone moves gracefully, revealing majestic mountain peaks in the distance, with soft clouds drifting by. Rivers weave through the terrain like silver threads, and vibrant wildflowers dot the fields. The scene is immersive, evoking a sense of wonder and tranquility." # @param {"type":"string"}
positive_prompt = "Generate me a fast-paced, high-yield \"One-Shot Revision\" educational video script and B-roll video generation prompts based on the following content:00:00 - Introduction & Sources of LightWelcome back, champions! Today we are cracking Light in one direct shot! Light is a form of energy that produces the sensation of vision. We categorize objects into Luminous (emit their own light, like the Sun, stars, candles, and glowing filaments) and Non-Luminous (reflect light into our eyes, like books, tables, and the Moon). When non-luminous objects are heated above $800^\\circ\\text{C}$, they become incandescent and start glowing. IMPORTANT EXAM WARNING: A battery or dry cell is a source of electrical energy, NOT a luminous object until it powers a bulb filament to heat up!   Visual Cue (LTX Video Prompt): A slow macro tracking camera pan across a glowing tungsten bulb filament radiating intense white heat next to non-luminous dark wooden surfaces, rich dark laboratory background, volumetric warm glowing light, high-contrast dark amber and gold mood, crisp focus, photorealistic, 8k.02:15 - Reflection of Light & Image TypesReflection is the bouncing back of light rays from a polished surface. Silver is the absolute best reflector. Two non-negotiable laws apply: first, the incident ray, reflected ray, and normal all lie in the same plane; second, the angle of incidence equals the angle of reflection ($\\angle i = \\angle r$). We have two image types: Real Images (can be captured on a screen, formed by actual intersection of reflected rays, always inverted, e.g., cinema screens) and Virtual Images (cannot be captured on a screen, rays only appear to meet behind the mirror, always erect). Light exhibits a dual nature—particle nature explains shadows and reflection, while wave nature explains interference. IMPORTANT EXAM WARNING: When light strikes normally ($\\angle i = 0^\\circ$), it reflects back along the exact same path ($\\angle r = 0^\\circ$)!   Visual Cue (LTX Video Prompt): A cinematic side-dolly shot across a laser beam striking a pristine silver mirror surface at a 45-degree angle, bouncing off symmetrically along the normal line, sleek modern physics lab setting, sharp volumetric laser light rays, cool cyan and vivid red mood, crisp focus, photorealistic, 8k.05:00 - Image Formation by Plane MirrorPlane mirrors form virtual, erect, and equal-sized images where object distance equals image distance ($d_o = d_i$). However, they undergo Lateral Inversion—the left side appears right and right appears left. This is why AMBULANCE is written laterally inverted on the front of emergency vehicles, so drivers view it correctly in rear-view mirrors. Key uses include household mirrors, barber shops, periscopes, and kaleidoscopical systems. IMPORTANT EXAM WARNING: In multiple reflection questions, if two plane mirrors are placed at $90^\\circ$, the number of images formed is $n = \\frac{360^\\circ}{90^\\circ} - 1 = 3$.   Visual Cue (LTX Video Prompt): A slow forward tracking push-in shot on a clear glass periscope and an ambulance grille with laterally inverted red lettering reflected in a polished rearview mirror, rain-slicked suburban street background, realistic evening street lighting, deep blue and vibrant red contrast mood, crisp focus, photorealistic, 8k.07:30 - Spherical Mirrors & Key DefinitionsSpherical mirrors are curved: Concave curves inward (converging), while Convex curves outward (diverging). Memorize these core terms: the Pole ($P$) is the center of the mirror surface; the Centre of Curvature ($C$) is the center of the sphere the mirror is cut from; the Principal Focus ($F$) is the point where parallel rays converge or appear to diverge; and the Focal Length ($f$) is the distance between $P$ and $F$. The golden relation is $f = \\frac{R}{2}$. IMPORTANT EXAM WARNING: Radius of curvature is always double the focal length ($R = 2f$). If $R = 30\\text{ cm}$, $f$ is strictly $15\\text{ cm}$!   Visual Cue (LTX Video Prompt): A slow orbital camera rotation around a shiny chrome curved concave mirror reflecting parallel light rays converging precisely at a single focal point, dark polished optics workstation setup, dramatic cinematic rim lighting, deep midnight blue and silver metallic mood, crisp focus, photorealistic, 8k.10:45 - Concave/Convex Ray Rules, Mirror Formula & MagnificationMaster these concave mirror image positions: an object at Infinity produces an image at $F$ (highly diminished, real); an object Beyond $C$ produces an image between $F$ and $C$ (diminished, real); an object at $C$ produces an image at $C$ (same size, real); an object between $F$ and $C$ produces an image beyond $C$ (enlarged, real); an object at $F$ produces an image at Infinity (highly enlarged, real); and an object between $P$ and $F$ produces an image behind the mirror (enlarged, virtual & erect). The Mirror Formula is $\\frac{1}{f} = \\frac{1}{v} + \\frac{1}{u}$, and Magnification is $M = -\\frac{v}{u} = \\frac{h_i}{h_o}$. Concave mirrors are used in headlights, shaving mirrors, and dentist tools, while Convex mirrors are used as driver rear-view mirrors due to their wider field of view. IMPORTANT EXAM WARNING: Concave mirrors produce a virtual and erect image in only ONE case—when the object is placed between Pole ($P$) and Focus ($F$)!   Visual Cue (LTX Video Prompt): A smooth horizontal dolly camera shot across a glowing dental mirror examining a detailed tooth model, alongside a solar furnace dish focusing intense sunlight, high-tech clinical environment, bright directional sunlight and focused rays, clinical white and vivid golden highlight mood, crisp focus, photorealistic, 8k.14:15 - Refraction of Light & Snell's LawRefraction is the bending of light when passing from one medium to another due to a change in speed. The speed of light in a vacuum/air is $3 \\times 10^8\\text{ m/s}$, in water is $2.25 \\times 10^8\\text{ m/s}$, and in glass is $2 \\times 10^8\\text{ m/s}$. Moving from a Rarer to Denser medium bends light towards the normal, whereas moving from a Denser to Rarer medium bends light away from the normal. Snell's Law defines refractive index: $\\mu = \\frac{\\sin i}{\\sin r}$. Real-world effects include a pencil appearing bent in water, a swimming pool floor appearing shallower, and atmospheric mirages.   Visual Cue (LTX Video Prompt): A slow macro pan across a clear glass cup filled with water showing a silver spoon sharply bent at the air-water interface, dark polished mahogany table setting, soft diffused sunlight streaming through a glass window, natural warm earth tone mood, crisp focus, photorealistic, 8k.17:30 - Lenses & Power of LensLenses are transparent media bounded by two spherical surfaces. A Convex Lens is converging (thicker in the center), while a Concave Lens is diverging (thinner in the center). Power of Lens ($P$) measures the degree of convergence or divergence, defined as $P = \\frac{1}{f\\text{ (in meters)}}$. Its S.I. unit is the Dioptre ($D$). Power for a convex lens is positive ($+$) and for a concave lens is negative ($-$). For multiple lenses in contact, $P_{net} = P_1 + P_2 + \\dots + P_n$. IMPORTANT EXAM WARNING: Always convert focal length into meters before calculating power! If $f = 50\\text{ cm} = 0.5\\text{ m}$, then $P = \\frac{1}{0.5} = +2\\text{ D}$. Combining $+4.5\\text{ D}$ and $-3.5\\text{ D}$ yields $P = +1\\text{ D}$ and $f = 1\\text{ m}$!   Visual Cue (LTX Video Prompt): A slow macro tracking shot through a double convex glass lens focusing parallel laser beams to a pinpoint focus on a dark target board, clean optical bench background, intense blue laser volumetric beams, high-tech neon blue and deep slate mood, crisp focus, photorealistic, 8k.21:00 - Dispersion of Light & Prism SpectrumDispersion is the splitting of white light into its 7 constituent colors (VIBGYOR) when passing through a glass prism. This occurs because different wavelengths travel at different speeds and refract at different angles through glass. Red light has the longest wavelength and refracts the least (top of spectrum), while Violet has the shortest wavelength and refracts the most (bottom of spectrum). Rainbows are formed by dispersion through suspended raindrops after rain. IMPORTANT EXAM WARNING: In exam MCQs, remember: Red light = minimum refraction/deviation; Violet light = maximum refraction/deviation!   Visual Cue (LTX Video Prompt): A cinematic slow zoom-in shot on a triangular glass prism refracting a thin beam of white light into a vibrant seven-color spectrum rainbow band on a dark screen, dark dusty physics room setting, intense volumetric light beams, brilliant rainbow gradient and dark shadow contrast mood, crisp focus, photorealistic, 8k.24:15 - Atmospheric Refraction & Scattering of LightAtmospheric Refraction occurs due to varying optical densities in Earth's atmospheric layers. Phenomena include the twinkling of stars as well as advanced sunrise and delayed sunset. The Sun is visible 2 minutes before actual sunrise and 2 minutes after actual sunset, adding 4 minutes to total daylight duration! Meanwhile, Scattering of Light occurs when sunlight strikes atmospheric gas molecules, where shorter wavelengths (Blue) scatter far more intensely than longer wavelengths (Red). This scattered blue light fills the sky, making it appear blue to our eyes. Lastly, sunlight takes approximately 8 minutes and 17 seconds to travel from the Sun to Earth.   Visual Cue (LTX Video Prompt): A wide sweeping camera pan across the blue Earth atmosphere during sunset showing light rays bending through atmosphere layers toward the horizon, view from low Earth orbit with stars in background, warm golden sunset light and deep atmospheric blue scattered haze, cosmic contrast mood, crisp focus, photorealistic, 8k." # @param {"type":"string"}
negative_prompt = "low quality, worst quality, deformed, distorted, disfigured, motion smear, motion artifacts, fused fingers, bad anatomy, weird hand, ugly" # @param {"type":"string"}
width = 832 # @param {"type":"number"}
height = 480 # @param {"type":"number"}
seed = 0 # @param {"type":"integer"}
steps = 25 # @param {"type":"integer", "min":1, "max":100}
cfg_scale = 2.05 # @param {"type":"number", "min":1, "max":20}
sampler_name = "res_multistep" # @param ["res_multistep", "euler", "dpmpp_2m", "ddim", "lms"]
frames = 73 # @param {"type":"integer", "min":1, "max":120}
fps = 24 # @param {"type":"integer", "min":1, "max":60}

with torch.inference_mode():
    generate_video(
        positive_prompt=positive_prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        seed=seed,
        steps=steps,
        cfg_scale=cfg_scale,
        sampler_name=sampler_name,
        length=frames,
        fps=fps
    )
clear_memory()